# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [24]:
# Write your code below.

%reload_ext dotenv
%dotenv


In [25]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [26]:
import os
from glob import glob
import pandas as pd

# Write your code below.
os.getenv("PRICE_DATA")

parquet_price_data = glob(os.path.join(os.getenv("PRICE_DATA"), "**/*.parquet"), recursive=True)



For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.


#using the dask_dataframe
dask_dd = dd.read_parquet(os.path.join(os.getenv("PRICE_DATA"), "**/*.parquet"),recursive = True)


dd_shift = (
    dask_dd
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending = True)
                       .assign(Close_lag_1 = x['Close'].shift(1),
                               Adj_lag_1 = x['Adj Close'].shift(1), 
                               hi_lo_range = x['High']-x['Low']),
            
            meta = pd.DataFrame(data ={'Date': 'datetime64[ns]',
                    'Open': 'f8',
                    'High': 'f8',
                    'Low': 'f8',
                    'Close': 'f8',
                    'Adj Close': 'f8',
                    'Volume': 'i8',
                    'source': 'object',
                    'Year': 'int32',
                    'Close_lag_1': 'f8',
                    'Adj_lag_1':'f8',
                    'hi_lo_range':'f8'},
                    index = pd.Index([], dtype=pd.StringDtype(), name='ticker'))
    ,include_groups = False).assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1)
)
    
    




+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [55]:
dd_feat = dd_shift.compute()

In [56]:
dd_feat['Moving_Average_10_DAYS'] = dd_feat['Returns'].rolling(10).mean()
dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_lag_1,hi_lo_range,Returns,Moving_Average_10_DAYS
0,1994-03-23,6.500000,8.000000,6.500000,7.750000,7.750000,952400.0,MPAA.csv,1994,6.060000,6.060000,1.500000,0.278878,NaN
1,1994-03-24,7.750000,9.250000,7.500000,8.500000,8.500000,312100.0,MPAA.csv,1994,7.750000,7.750000,1.750000,0.096774,NaN
2,1994-03-25,8.875000,9.000000,8.250000,8.625000,8.625000,282000.0,MPAA.csv,1994,8.500000,8.500000,0.750000,0.014706,NaN
3,1994-03-28,8.250000,8.875000,8.000000,8.000000,8.000000,85700.0,MPAA.csv,1994,8.625000,8.625000,0.875000,-0.072464,NaN
4,1994-03-29,8.000000,8.250000,7.625000,8.250000,8.250000,78100.0,MPAA.csv,1994,8.000000,8.000000,0.625000,0.031250,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9221,2020-03-26,60.169998,66.139999,60.020000,65.660004,65.660004,3445300.0,PAYX.csv,2020,60.080002,60.080002,6.119999,0.092876,0.004374
9222,2020-03-27,62.549999,63.990002,59.680000,60.169998,60.169998,4114300.0,PAYX.csv,2020,65.660004,65.660004,4.310001,-0.083613,-0.011661
9223,2020-03-30,60.680000,64.949997,60.279999,64.419998,64.419998,3754700.0,PAYX.csv,2020,60.169998,60.169998,4.669998,0.070633,0.014883
9224,2020-03-31,63.900002,64.849998,61.410000,62.919998,62.919998,5368500.0,PAYX.csv,2020,64.419998,64.419998,3.439999,-0.023285,0.007053


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No it wasn't necessary to convert to pandas to calculate the moving average 

Yes it would be better for faster calculations although there might be partition errors

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.